In [1]:
# NOTEBOOK 24 - build the Kolkata and Mumbai datasets in exactly the format the Delhi v3 models expect
# Source: OpenAQ archive (CPCB-network hourly data incl. temperature and humidity) + Open-Meteo ERA5 wind.
# Output: crosscity_cache/city_<name>.npz with X_raw (T,N,26), Y_raw, era5 wind, dynamic A(t), static A.
# No Delhi file is touched. Run notebook 23 first (it creates crosscity_stations.csv).
# HOW TO CHECK: this cell prints the config.
import os, io, json, time, math, datetime as dt
import numpy as np, pandas as pd, requests
from concurrent.futures import ThreadPoolExecutor

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
WORK   = os.path.join(V2_DIR, 'crosscity_cache')
MODEL_DIR = os.path.join(V2_DIR, 'ecotwin_outputs', 'models')
S3 = 'https://openaq-data-archive.s3.amazonaws.com/records/csv.gz'
CITIES = ['Kolkata', 'Mumbai', 'Ahmedabad', 'Patna']
DATA_START, DATA_END = '2025-09-01', '2026-04-10'     # Delhi test period is 8 Dec 2025 - 9 Apr 2026; Sep-Nov is for few-shot
THRESHOLD_KM = 12.0
cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']
GRID = pd.date_range(DATA_START, DATA_END + ' 00:00', freq='h')
print('cities', CITIES, '| hourly grid', len(GRID), GRID[0], '->', GRID[-1], '| features', len(FEATURE_COLS))

cities ['Kolkata', 'Mumbai', 'Ahmedabad', 'Patna'] | hourly grid 5305 2025-09-01 00:00:00 -> 2026-04-10 00:00:00 | features 26


In [2]:
# 1) usable stations from the notebook-23 audit
st = pd.read_csv(os.path.join(WORK, 'crosscity_stations.csv'))
st = st[st['usable'] & st['city'].isin(CITIES)].reset_index(drop=True)
print(st.groupby('city').size().to_dict())

# 2) hourly values per station from the archive (15-minute records -> hourly means), cached per station
PARAMS = {'pm25': 'pm2.5', 'pm10': 'pm10', 'no2': 'no2', 'so2': 'so2', 'co': 'co', 'o3': 'o3',
          'temperature': 'at', 'relativehumidity': 'rh'}
def read_day(url):
    try:
        r = requests.get(url, timeout=40)
        if r.status_code != 200:
            return None
        d = pd.read_csv(io.BytesIO(r.content), compression='gzip', usecols=['datetime', 'parameter', 'units', 'value'])
        d = d[d['parameter'].isin(PARAMS)]
        return d if len(d) else None
    except Exception:
        return None

def to_delhi_units(d):
    # The archive labels CPCB gases 'ppb', but the numbers are already in CPCB's own units: CO about 1.3 is mg/m3
    # (real ambient CO is hundreds of ppb), and no2/so2 line up with Delhi's ug/m3 scale without conversion.
    # So the values are used as they are (checked against Delhi's means in the last cell).
    return d.copy()

def fetch_station(loc_id):
    f = os.path.join(WORK, f'hourly_{loc_id}.pkl')
    if os.path.exists(f):
        return pd.read_pickle(f)
    urls = []
    day = dt.date.fromisoformat(DATA_START) - dt.timedelta(days=1)
    end = dt.date.fromisoformat(DATA_END) + dt.timedelta(days=1)
    while day <= end:
        urls.append(f'{S3}/locationid={loc_id}/year={day.year}/month={day.month:02d}/location-{loc_id}-{day:%Y%m%d}.csv.gz')
        day += dt.timedelta(days=1)
    with ThreadPoolExecutor(max_workers=8) as ex:
        parts = [p for p in ex.map(read_day, urls) if p is not None]
    if not parts:
        out = pd.DataFrame(index=GRID, columns=list(PARAMS.values()), dtype=float)
    else:
        d = to_delhi_units(pd.concat(parts))
        d['t'] = pd.to_datetime(d['datetime'], utc=True).dt.tz_convert('Asia/Kolkata').dt.tz_localize(None).dt.floor('h')
        d['name'] = d['parameter'].map(PARAMS)
        out = d.pivot_table(index='t', columns='name', values='value', aggfunc='mean').reindex(GRID)
        for c in PARAMS.values():
            if c not in out.columns:
                out[c] = np.nan
    out.to_pickle(f)
    return out

t0 = time.time()
hourly = {}
for k, r in st.iterrows():
    hourly[int(r.location_id)] = fetch_station(int(r.location_id))
    if (k + 1) % 5 == 0:
        print(f'  {k+1}/{len(st)} stations  {time.time()-t0:.0f}s')
print(f'archive read in {time.time()-t0:.0f}s')

# 2b) keep stations whose PM2.5 is mostly really measured; for stations whose humidity/temperature sensor is mostly
#     missing, those two variables are taken from ERA5 instead of being imputed (done after the wind fetch below)
MIN_PM, MIN_RHAT = 0.80, 0.70
CHECK = {'pm2.5': (0, 1000), 'rh': (0, 100.0001), 'at': (-10, 55)}
def share_ok(h, col):
    lo, hi = CHECK[col]
    return float(((h[col] >= lo) & (h[col] < hi)).mean())
keep = np.array([share_ok(hourly[int(r.location_id)], 'pm2.5') >= MIN_PM for _, r in st.iterrows()])
print('dropped (PM2.5 measured less than 80 % of the hours):')
for _, r in st[~keep].iterrows():
    print(f"  {r.city:8} {r['name'][:34]:34} pm2.5 {share_ok(hourly[int(r.location_id)], 'pm2.5'):.2f}")
st = st[keep].reset_index(drop=True)
WEAK = {int(r.location_id): (share_ok(hourly[int(r.location_id)], 'rh') < MIN_RHAT or share_ok(hourly[int(r.location_id)], 'at') < MIN_RHAT)
        for _, r in st.iterrows()}
print('stations kept per city:', st.groupby('city').size().to_dict(), '| humidity/temperature from ERA5 for', sum(WEAK.values()), 'stations')
assert (st.groupby('city').size() >= 6).all(), 'fewer than 6 stations left in a city'


# HOW TO CHECK: about 5-10 minutes the first time; instant when re-run (cached in crosscity_cache/hourly_*.pkl).

{'Ahmedabad': 9, 'Kolkata': 10, 'Mumbai': 27, 'Patna': 7}
  5/53 stations  0s
  10/53 stations  0s
  15/53 stations  0s
  20/53 stations  0s
  25/53 stations  0s
  30/53 stations  0s
  35/53 stations  0s
  40/53 stations  0s
  45/53 stations  1s
  50/53 stations  1s
archive read in 1s
dropped (PM2.5 measured less than 80 % of the hours):
  Mumbai   Borivali East, Mumbai - MPCB       pm2.5 0.78
  Mumbai   Sion, Mumbai - MPCB                pm2.5 0.74
  Mumbai   Khadakpada, Kalyan - MPCB          pm2.5 0.65
  Mumbai   Mulund West, Mumbai - MPCB         pm2.5 0.77
  Mumbai   Powai, Mumbai - MPCB               pm2.5 0.75
  Mumbai   Vile Parle West, Mumbai - MPCB     pm2.5 0.71
  Patna    IGSC Planetarium Complex, Patna -  pm2.5 0.74
stations kept per city: {'Ahmedabad': 9, 'Kolkata': 10, 'Mumbai': 21, 'Patna': 6} | humidity/temperature from ERA5 for 5 stations


In [3]:
# 3) ERA5 wind at each station (same source and u/v convention as notebook 13), cached
def fetch_wind(lat, lon, retries=4):
    params = {'latitude': lat, 'longitude': lon, 'start_date': DATA_START, 'end_date': DATA_END,
              'hourly': 'wind_speed_10m,wind_direction_10m', 'wind_speed_unit': 'ms', 'timezone': 'Asia/Kolkata'}
    for k in range(retries):
        r = requests.get('https://archive-api.open-meteo.com/v1/archive', params=params, timeout=90)
        if r.status_code == 200:
            h = r.json()['hourly']
            return pd.DataFrame({'ws': h['wind_speed_10m'], 'wd': h['wind_direction_10m']}, index=pd.DatetimeIndex(h['time'])).reindex(GRID)
        time.sleep(8 * (k + 1))
    raise RuntimeError('Open-Meteo failed: ' + r.text[:200])

wind = {}
for k, r in st.iterrows():
    f = os.path.join(WORK, f'era5_{int(r.location_id)}.pkl')
    if os.path.exists(f):
        wind[int(r.location_id)] = pd.read_pickle(f); continue
    wind[int(r.location_id)] = fetch_wind(r.lat, r.lon)
    wind[int(r.location_id)].to_pickle(f)
    time.sleep(1.0)
print('ERA5 wind ready for', len(wind), 'stations')

# ERA5 temperature and humidity for the stations whose own sensors are mostly missing (replaces the whole series for that station)
def fetch_tr(lat, lon, retries=4):
    params = {'latitude': lat, 'longitude': lon, 'start_date': DATA_START, 'end_date': DATA_END,
              'hourly': 'temperature_2m,relative_humidity_2m', 'timezone': 'Asia/Kolkata'}
    for k in range(retries):
        r = requests.get('https://archive-api.open-meteo.com/v1/archive', params=params, timeout=90)
        if r.status_code == 200:
            h = r.json()['hourly']
            return pd.DataFrame({'at': h['temperature_2m'], 'rh': h['relative_humidity_2m']}, index=pd.DatetimeIndex(h['time'])).reindex(GRID)
        time.sleep(8 * (k + 1))
    raise RuntimeError('Open-Meteo failed: ' + r.text[:200])
for k, r in st.iterrows():
    lid = int(r.location_id)
    if not WEAK[lid]:
        continue
    f = os.path.join(WORK, f'era5tr_{lid}.pkl')
    if os.path.exists(f):
        tr = pd.read_pickle(f)
    else:
        tr = fetch_tr(r.lat, r.lon); tr.to_pickle(f); time.sleep(1.0)
    hourly[lid] = hourly[lid].copy(); hourly[lid]['at'] = tr['at'].values; hourly[lid]['rh'] = tr['rh'].values
print('ERA5 temperature/humidity applied to', sum(WEAK.values()), 'stations')

# HOW TO CHECK: about 1-2 minutes (one call per station, 1 s apart).

ERA5 wind ready for 46 stations
ERA5 temperature/humidity applied to 5 stations


In [4]:
# 4) helpers: Delhi-style cleaning, tiered imputation, features and graphs
RANGES = {'pm2.5': (0, 1000), 'pm10': (0, 2000), 'no2': (0, 500), 'so2': (0, 500), 'co': (0, 50),
          'o3': (0, 500), 'rh': (0, 100), 'at': (-10, 55)}          # out of range -> missing BEFORE imputation (like the Delhi pipeline)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def geometry(lats, lons):
    n = len(lats)
    dist = np.zeros((n, n)); du = np.zeros((n, n)); dv = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                dist[i, j] = haversine_km(lats[i], lons[i], lats[j], lons[j])
                de = (lons[j] - lons[i]) * np.cos(np.radians(lats[i])); dn = lats[j] - lats[i]; nr = np.hypot(de, dn)
                if nr > 0:
                    du[i, j], dv[i, j] = de / nr, dn / nr
    within = (dist > 0) & (dist <= THRESHOLD_KM)
    return dist, du, dv, within

def impute(P, A):
    # P (T,N) with NaN. tier 1 spatial neighbour mean, 2 city mean at that hour, 3 linear interpolation, 4 ffill/bfill, 5 global median
    P = P.copy(); miss0 = np.isnan(P)
    valid = (~np.isnan(P)).astype(float); v0 = np.nan_to_num(P)
    num = v0 @ A.T; den = valid @ A.T
    fill = np.where(den > 0, num / np.maximum(den, 1e-9), np.nan)
    P = np.where(np.isnan(P), fill, P)
    cm = np.nanmean(P, axis=1, keepdims=True) if np.isfinite(P).any() else np.zeros((P.shape[0], 1))
    P = np.where(np.isnan(P), np.broadcast_to(cm, P.shape), P)
    df = pd.DataFrame(P).interpolate(limit_direction='both').ffill().bfill()
    P = df.values
    P = np.where(np.isnan(P), np.nanmedian(P) if np.isfinite(P).any() else 0.0, P)
    return P.astype(np.float32), miss0

def compute_At_vec(u, v, du, dv, dist, within, eps=1e-6):
    align = u[:, None] * du + v[:, None] * dv
    A = np.where(within & (align > 0.0), align / (dist + eps), 0.0).astype(np.float32)
    rs = A.sum(1, keepdims=True); rs[rs == 0] = 1.0
    return A / rs
print('helpers ready')

helpers ready


In [5]:
# 5) build one dataset per city and save it
summary = {}
for city in CITIES:
    s = st[st['city'] == city].reset_index(drop=True)
    ids = [int(x) for x in s['location_id']]
    names = s['name'].tolist(); lats = s['lat'].values.astype(float); lons = s['lon'].values.astype(float)
    n = len(ids); T = len(GRID)
    dist, du, dv, within = geometry(lats, lons)
    static_adj = within.astype(np.float32)                         # symmetric 0/1 distance graph, no self loops (as in Delhi)

    panel, obs_mask, imputed_share = {}, None, {}
    for col in ['pm2.5', 'pm10', 'no2', 'so2', 'co', 'o3', 'rh', 'at']:
        P = np.stack([hourly[i][col].values.astype(float) for i in ids], axis=1)
        lo, hi = RANGES[col]
        P = np.where((P >= lo) & (P < hi), P, np.nan)
        P, miss = impute(P, static_adj)
        panel[col] = P; imputed_share[col] = float(miss.mean())
        if col == 'pm2.5':
            obs_mask = ~miss                                        # True where PM2.5 was really observed
    uv = np.zeros((T, n, 2), dtype=np.float32)
    for k, i in enumerate(ids):
        w = wind[i]; wd = np.deg2rad(w['wd'].values.astype(float) % 360); ws = w['ws'].values.astype(float)
        uv[:, k, 0] = np.nan_to_num(-ws * np.sin(wd)); uv[:, k, 1] = np.nan_to_num(-ws * np.cos(wd))
    spd = np.hypot(uv[..., 0], uv[..., 1])

    pm = pd.DataFrame(panel['pm2.5'])
    f = {}
    f['pm_roll6_mean'] = pm.rolling(6, min_periods=1).mean().values
    f['pm_roll6_std'] = pm.rolling(6, min_periods=1).std().fillna(0.0).values
    f['pm_roll24_mean'] = pm.rolling(24, min_periods=1).mean().values
    f['pm_roll24_std'] = pm.rolling(24, min_periods=1).std().fillna(0.0).values
    f['pm_delta3'] = (pm - pm.shift(3)).fillna(0.0).values
    cmean = pm.mean(axis=1).values[:, None]
    f['city_mean_pm'] = np.broadcast_to(cmean, (T, n)); f['city_max_pm'] = np.broadcast_to(pm.max(axis=1).values[:, None], (T, n))
    f['pm_minus_city'] = pm.values - cmean
    hh, dow, mon = GRID.hour.values, GRID.dayofweek.values, GRID.month.values
    tcols = {'hour_sin': np.sin(2*np.pi*hh/24), 'hour_cos': np.cos(2*np.pi*hh/24), 'dow_sin': np.sin(2*np.pi*dow/7),
             'dow_cos': np.cos(2*np.pi*dow/7), 'mon_sin': np.sin(2*np.pi*(mon-1)/12), 'mon_cos': np.cos(2*np.pi*(mon-1)/12),
             'is_weekend': (dow >= 5).astype(float)}
    for k2, v2 in tcols.items():
        f[k2] = np.broadcast_to(v2[:, None], (T, n))

    X = np.zeros((T, n, len(FEATURE_COLS)), dtype=np.float32)
    for fi, col in enumerate(FEATURE_COLS):
        if col in panel:            X[:, :, fi] = panel[col]
        elif col == 'u_wind':       X[:, :, fi] = uv[..., 0]
        elif col == 'v_wind':       X[:, :, fi] = uv[..., 1]
        elif col == 'wspd':         X[:, :, fi] = spd
        else:                       X[:, :, fi] = f[col]
    assert not np.isnan(X).any(), 'NaN left in features'

    dyn = np.stack([compute_At_vec(uv[t, :, 0], uv[t, :, 1], du, dv, dist, within) for t in range(T)]).astype(np.float32)
    np.savez_compressed(os.path.join(WORK, f'city_{city}.npz'), X_raw=X, Y_raw=X[:, :, 0].copy(), obs_mask=obs_mask, era5=uv,
                        dyn_adj=dyn, static_adj=static_adj, names=np.array(names, dtype=str), lats=lats, lons=lons, grid=np.array(GRID.strftime('%Y-%m-%d %H:%M:%S'), dtype='U19'))
    summary[city] = {'stations': n, 'hours': T, 'pairs_within_12km': int(within.sum() // 2), 'edges_per_hour': float((dyn > 0).sum((1, 2)).mean()),
                     'mean_pm25': float(X[:, :, 0].mean()), 'pm25_observed_share': float(obs_mask.mean()),
                     'imputed_share': {k: round(v, 3) for k, v in imputed_share.items()}}
    print(city, json.dumps(summary[city], indent=1))
json.dump(summary, open(os.path.join(WORK, 'city_build_summary.json'), 'w'), indent=2)

# HOW TO CHECK: for each city pm25_observed_share is about 0.85-0.92 (matches the audit), edges_per_hour is
# in the tens to low hundreds, mean_pm25 is plausible (Kolkata winter ~70-100, Mumbai ~40-60).

C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_13232\2338478364.py:31: RuntimeWarning: Mean of empty slice
  cm = np.nanmean(P, axis=1, keepdims=True) if np.isfinite(P).any() else np.zeros((P.shape[0], 1))
C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_13232\2753916126.py:36: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  hh, dow, mon = GRID.hour.values, GRID.dayofweek.values, GRID.month.values


Kolkata {
 "stations": 10,
 "hours": 5305,
 "pairs_within_12km": 37,
 "edges_per_hour": 37.31347785108388,
 "mean_pm25": 66.14716339111328,
 "pm25_observed_share": 0.8913289349670123,
 "imputed_share": {
  "pm2.5": 0.109,
  "pm10": 0.109,
  "no2": 0.109,
  "so2": 0.113,
  "co": 0.118,
  "o3": 0.109,
  "rh": 0.116,
  "at": 0.109
 }
}


C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_13232\2338478364.py:31: RuntimeWarning: Mean of empty slice
  cm = np.nanmean(P, axis=1, keepdims=True) if np.isfinite(P).any() else np.zeros((P.shape[0], 1))
C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_13232\2753916126.py:36: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  hh, dow, mon = GRID.hour.values, GRID.dayofweek.values, GRID.month.values


Mumbai {
 "stations": 21,
 "hours": 5305,
 "pairs_within_12km": 53,
 "edges_per_hour": 53.85956644674835,
 "mean_pm25": 43.09226989746094,
 "pm25_observed_share": 0.8651586553565819,
 "imputed_share": {
  "pm2.5": 0.135,
  "pm10": 0.133,
  "no2": 0.152,
  "so2": 0.234,
  "co": 0.18,
  "o3": 0.151,
  "rh": 0.116,
  "at": 0.117
 }
}


C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_13232\2338478364.py:31: RuntimeWarning: Mean of empty slice
  cm = np.nanmean(P, axis=1, keepdims=True) if np.isfinite(P).any() else np.zeros((P.shape[0], 1))
C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_13232\2753916126.py:36: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  hh, dow, mon = GRID.hour.values, GRID.dayofweek.values, GRID.month.values


Ahmedabad {
 "stations": 9,
 "hours": 5305,
 "pairs_within_12km": 23,
 "edges_per_hour": 23.73722902921772,
 "mean_pm25": 52.651546478271484,
 "pm25_observed_share": 0.8819981149858624,
 "imputed_share": {
  "pm2.5": 0.118,
  "pm10": 0.118,
  "no2": 0.119,
  "so2": 0.12,
  "co": 0.12,
  "o3": 0.12,
  "rh": 0.119,
  "at": 0.12
 }
}
Patna {
 "stations": 6,
 "hours": 5305,
 "pairs_within_12km": 9,
 "edges_per_hour": 8.92177191328935,
 "mean_pm25": 58.93235778808594,
 "pm25_observed_share": 0.8630851398052152,
 "imputed_share": {
  "pm2.5": 0.137,
  "pm10": 0.138,
  "no2": 0.137,
  "so2": 0.142,
  "co": 0.146,
  "o3": 0.141,
  "rh": 0.136,
  "at": 0.136
 }
}


C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_13232\2338478364.py:31: RuntimeWarning: Mean of empty slice
  cm = np.nanmean(P, axis=1, keepdims=True) if np.isfinite(P).any() else np.zeros((P.shape[0], 1))
C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_13232\2753916126.py:36: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  hh, dow, mon = GRID.hour.values, GRID.dayofweek.values, GRID.month.values


In [6]:
# 6) sanity: compare the new cities' feature statistics with Delhi's (a large mismatch warns about unit problems)
import pickle
fs = pickle.load(open(os.path.join(V2_DIR, 'ecotwin_outputs', 'scalers', 'v3_feature_scaler.pkl'), 'rb'))
print(f"{'feature':16}{'Delhi mean':>12}" + ''.join(f'{c + " mean":>16}' for c in CITIES))
D = {c: np.load(os.path.join(WORK, f'city_{c}.npz'))['X_raw'] for c in CITIES}
for fi, col in enumerate(FEATURE_COLS):
    if col in ('pm2.5','pm10','no2','so2','co','o3','rh','at','u_wind','v_wind','wspd'):
        print(f'{col:16}{fs.mean_[fi]:>12.2f}' + ''.join(f'{D[c][:, :, fi].mean():>16.2f}' for c in CITIES))

# HOW TO READ: Delhi values are averages over 2024-2026. no2/so2/co are the units most at risk (ppb -> ug/m3 or mg/m3).
# If a unit looks wrong by 10x or more, tell me before running the evaluation.

feature           Delhi mean    Kolkata mean     Mumbai mean  Ahmedabad mean      Patna mean
pm2.5                  95.51           66.15           43.09           52.65           58.93
pm10                  201.75          124.47          104.97          127.85          127.83
no2                    43.17           35.62           31.71           39.70           24.91
so2                    14.70           10.16            7.86           20.22            6.16
co                      1.23            0.97            0.69            0.80            1.13
o3                     36.93           25.32           31.42           30.09           27.75
rh                     61.71           70.96           61.08           54.57           75.13
at                     25.93           24.58           26.55           24.84           21.96
u_wind                  0.30            0.16            0.39           -0.17            0.92
v_wind                 -0.41           -0.34           -0.52          

C:\Users\jlakk_j7eh5fd\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
